# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and leverages a rich schema with explicit `@id` references for all entities.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}  |  Version: {getattr(metadata, 'version', None)}")
print(f"DOI: {getattr(metadata, 'identifier', None)}\n")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The FAIR² Croissant schema organizes data into record sets (tables). Fields in each set are columns. We'll enumerate all record sets and their fields using their `@id` for reference.


In [ ]:
# List all available record sets and their fields using @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in this dataset schema. Please check the Croissant resource.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs.name} (@id={rs.id})")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id={field.id}) (type={field.data_type})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

In this example, we'll extract data from each available record set into a pandas DataFrame. Record set and field `@id`s are used for all references.


In [ ]:
# Collect @id for all record sets
record_set_ids = []
for rs in getattr(dataset.metadata, 'record_sets', []):
    record_set_ids.append(rs.id)

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set @id: {rs_id} ({len(df)} records, {df.shape[1]} columns)")

if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"First record set columns: {dataframes[example_rs_id].columns.tolist()}")
    display(dataframes[example_rs_id].head())
else:
    print("No record sets/dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare for further analysis.

We'll identify a record set containing numeric data for further analysis. All IDs are referenced by `@id`.

In [ ]:
# Identify numeric fields from the first record set for demonstration
import numpy as np

if record_set_ids:
    # Attempt to find numeric fields
    rs = dataset.metadata.record_sets[0]
    numeric_field_id = None
    for field in rs.fields:
        if str(field.data_type).lower() in ['float', 'integer', 'number', 'double']:
            numeric_field_id = field.id
            group_field_id = None
            # Also try to pick a group/categorical field
            for f in rs.fields:
                if f.id != numeric_field_id and (str(f.data_type).lower() in ['string', 'text'] or 'name' in f.name.lower()):
                    group_field_id = f.id
                    break
            break

    if numeric_field_id is not None:
        df = dataframes[rs.id]

        # Defensive conversion
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        print(f"Using numeric field @id: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()

        print(f"Filtered records where {numeric_field_id} > mean ({threshold:0.2f})")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized field {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a group_field if possible
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No obvious numeric field found for the first record set.")
else:
    print("No record sets available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We demonstrate a histogram and boxplot of a numeric variable (referenced by its `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`. We demonstrated how to extract metadata, enumerate schemas, load tables by `@id`, filter and analyze numeric data, and create basic visualizations. The explicit use of `@id` for referencing schemas ensures reproducibility and compliance with Croissant best practices.